# 05 — Model Explainability

This notebook computes permutation importance and SHAP values to explain model predictions.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from src.data_loader import load_raw_csv
from src.preprocessing import preprocess
from src.features import prepare_features, create_time_split
from src.train import run_training
from src.explainability import compute_permutation_importance, plot_permutation_importance, try_shap_explanation
from src.config import CLEANED_CSV

## 1. Load and Train Model

In [ ]:
if CLEANED_CSV.exists():
    df = pd.read_csv(CLEANED_CSV)
else:
    raw = load_raw_csv()
    df = preprocess(raw)

X, y = prepare_features(df)
X_train, X_test, y_train, y_test = create_time_split(X, y)
results, pipeline, preprocessor, best_name, paths = run_training(X_train, y_train, X_test, y_test)
print(f"Best model: {best_name}")

## 2. Permutation Importance

In [ ]:
importance_df = compute_permutation_importance(pipeline, X_test, y_test, n_repeats=10)
print(importance_df.head(15))

In [ ]:
plot_permutation_importance(importance_df, top_n=20)
plt.savefig('../reports/permutation_importance.png', dpi=150)
plt.show()

## 3. SHAP Analysis

In [ ]:
shap_path = try_shap_explanation(pipeline, X_test)
if shap_path:
    print(f"SHAP summary saved to: {shap_path}")
else:
    print("SHAP analysis not available. See permutation importance above.")

## 4. Interpretation

Key findings from explainability analysis:

1. **Permutation importance** shows which features most affect model predictions when shuffled
2. **SHAP values** (when available) show the direction and magnitude of each feature's contribution
3. **Feature importance indicates associations, not causation**
4. The model learns patterns from historical data, which may not generalize to future events

**Important:** These explanations describe what the model learned, not what causes funding outcomes.